Shopflow order events: make them, check them, turn them into bytes and back.

In [1]:
import json
from datetime import datetime, timedelta

In [2]:
PRODUCTS = {
    "P100": ("Mechanical keyboard", "electronics", 79.0),
    "P200": ("Running shoes", "sports", 120.0),
    "P300": ("Coffee beans", "grocery", 15.0),
    "P400": ("Yoga mats", "sports", 20.4),
    "P400": ("Milk", "grocery", 6.5),
}

CITIES = ["Manila", "Naga", "Cebu", "Batangas", "Sydney"]
START = datetime(2026, 11, 17, 23, 20)

SCHEMA = {"order_id": str, "customer_id": str, "product_id": str, "qty": int, "price": float, "city": str, "ts": str}

In [6]:
def make_order(n, rng):
    product_id = rng.choice(list(PRODUCTS))
    return {
        "schema_version": 1,
        "order_id": f"o-{n:05d}",
        "customer_id": f"c-{rng.randint(1, 40):03d}",
        "product_id": product_id,
        "qty": rng.randint(1, 3),
        "price": PRODUCTS[product_id][2],
        "city": rng.choice(CITIES),
        "ts": (START + timedelta(seconds=30 * n)).isoformat()        
    }

In [7]:
def validate(event):
    """
    Returns a list of problems. An empty list means an event is valid"""
    problems = []
    for field, kind in SCHEMA.items():
        if field not in event:
            problems.append(f"missing field {field}")
        elif not isinstance(event[field], kind):
            problems.append(f"{field} must be {kind.__name__}")
    
    if not problems and event["qty"] < 1:
        problems.append("qty must be at least 1")
    return problems

def encode(event):
    return json.dumps(event).encode("utf-8")

def decode(raw):
    return json.loads(raw.decode("utf-8"))

In [ ]:
# # Test part
# import random
# order = make_order(1, random.Random(1))
# print(order)
# print("problems:", validate(order))
# print("problems:", validate(dict(order, qty="two")))
# print("problems:", validate({"order_id": "o-1"}))
# print("bytes:", len(encode(order)))

{'schema_version': 1, 'order_id': 'o-00001', 'customer_id': 'c-037', 'product_id': 'P200', 'qty': 1, 'price': 120.0, 'city': 'Cebu', 'ts': '2026-11-17T23:20:30'}
problems: []
problems: ['qty must be int']
problems: ['missing field customer_id', 'missing field product_id', 'missing field qty', 'missing field price', 'missing field city', 'missing field ts']
bytes: 161


In [ ]:
# Test create orders
# > kafka-topics.sh --bootstrap-server $BS --create --topic orders --partitions 3